# Preparing Review Bundles
We present exactly five Review Bundles, each containing exactly two different CUAD categories, for a total of 10 categories. Each bundle contains:
- exactly 5 positive annotated spans, ie a span that answers the question for a category
- exactly 5 negative annotated spans, ie a span with text that does not answer the category question, but can look similar

## Importing training split
We select the categories from only the training split to avoid data leakage.

In [1]:
import json
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# import the training dataset and convert to a dataframe

cleaned_df = pd.read_csv('../data/cuad/cleaned_train_df.csv')
with open('../data/splits/train.json') as f:
    train_split = json.load(f)

# verify number of titles in training split against 
train_titles = {contract['title'] for contract in train_split['data']}

# the original cleaned_train_df may contain validation contracts too, so eliminate those:
CLEANED_DF_CONTAINS_LEAK = cleaned_df['title'].nunique() > len(train_titles)

if CLEANED_DF_CONTAINS_LEAK:
    print('Warning: cleaned_train_df contains contracts not in the train split.')
    print('Filtering the cleaned_train_df to only include contracts in the train split.')
    cleaned_train_df = cleaned_df[cleaned_df['title'].isin(train_titles)]

    # and verify that the cleaned_train_df contains the correct # of contracts
    assert(cleaned_train_df['title'].nunique() == len(train_titles))

    # export the filtered cleaned_df, and rename the original
    cleaned_df.to_csv('../data/cuad/cleaned_df.csv', index=False)
    cleaned_train_df.to_csv('../data/cuad/cleaned_train_df.csv', index=False)



## Selecting and grouping categories
We previously selected 10 core categories on the following metrics (more in 01_EDA.ipynb), but on the unfiltered dataset:

- **Class balance**: absolute number of answered (non-impossible) examples per category. A low impossibility rate alone doesn't automatically make a category a good candidate to focus on; categories can contain few impossible q/as but be infrequent overall. Absolute number of answerable examples balances frequency and impossibility. From the September milestone doc, we add the following requirement:
    - Each category has at least 20 positive Contracts and 20 positive context groups. Exceptions must be documented and justified.
    - **Positive contract definition**: the contract contains at least one instance of an answerable question of the selected category
    - **Positive context group definition:** clarify with teammates

- **Label quality/consistency**: coefficient of variation (std/mean) of `answer_text` length. Unlike variance and standard deviation, CV is a unitless measure that allows us to account for differences in mean answer/clause length and directly compare variation across categories. A high CV suggests annotators captured wildly different granularities under the same category (e.g. sometimes a single date, sometimes a whole clause), which is noisier to model and evaluate.

- **Span extraction difficulty**: median word count of `answer_text`. Very short spans (names, dates) are close to trivial extraction; very long spans are harder to extract cleanly, as they are liable to be broken across multiple chunks. Chunking for mid-length, or median clause-length, spans allows each chunk to contain clauses while not blurring multiple clauses into single units of information. Choosing categories with substantial-length, low-variation spans allows for simpler chunking strategies down the line.

- **Redundancy**: several categories cluster around the same legal concept (e.g. `Exclusivity` / `Non-Compete` / `Competitive Restriction Exception`, or `Cap On Liability` / `Uncapped Liability`, or the `License Grant` / `Non-Transferable License` / `Affiliate License` family). We prefer *two* strong representatives per cluster rather than several near-duplicates, to reduce noise and expand model versatility. Strong representatives display the above traits. We also consider semantic grouping of other important categories.

We alter the redundancy metric slightly, given that the [September milestone doc's]('../docs\coach/september-execution/Accenture-Sept_milestone.md') definition of a review bundle requires two related categories within each bundle. 

We add a few metrics from the [September milestone doc]('../docs\coach/september-execution/Accenture-Sept_milestone.md'):

Rerunning previous metric checks on the training set only:

### Analyzing metrics by category

In [24]:
# use only the train split contracts filtered above
df = cleaned_train_df.copy()

df['num_words'] = df['answer_text'].str.split().str.len()

category_stats = df.groupby('category').agg(
    n_answered=('has_answer', 'sum'),
    impossible_rate=('is_impossible', 'mean'),
    words_median=('num_words', 'median'),
    len_mean=('answer_len', 'mean'),
    len_std=('answer_len', 'std'),
).sort_values('n_answered', ascending=False)

category_stats['len_cv'] = category_stats['len_std'] / category_stats['len_mean']

# positive contract appearance: # of contracts where the contract contains at least one instance 
# of an answerable question of the selected category 
category_stats['positive_contract_appearances'] = df[df['has_answer']].groupby('category')['title'].nunique()
category_stats.round(2)



,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances
category,,,,,,,
Parties,1586,0.00,2.0,22.62,24.30,1.07,325
License Grant,513,0.24,52.0,423.93,294.12,0.69,164
Cap On Liability,432,0.25,48.0,363.50,230.50,0.63,181
Audit Rights,416,0.31,40.0,314.34,227.43,0.72,139
Anti-Assignment,404,0.18,34.5,302.33,256.15,0.85,235
Insurance,341,0.39,39.0,322.96,286.58,0.89,107
Document Name,335,0.00,3.0,29.95,19.45,0.65,326
Agreement Date,307,0.08,3.0,18.55,10.41,0.56,301
Expiration Date,302,0.17,31.0,233.44,155.90,0.67,264


In [25]:
# remove categories that fail the frequency/class balance checks
mask = (category_stats['n_answered'] < 100) | (category_stats['positive_contract_appearances'] < 20)
print('Number of categories with < 100 answerable instances or < 20 positive contract appearances:')
print(mask.sum())

category_stats = category_stats[~mask]
print('Number of categories remaining after filtering:')
print(category_stats.shape[0])


Number of categories with < 100 answerable instances or < 20 positive contract appearances:
14
Number of categories remaining after filtering:
27


### Comparing by answer length CV:

In [26]:
# view ordering by coefficient of variation of answer length``
category_stats.sort_values('len_cv', ascending=False).round(2)
# would be nice to have this as an interactive table for easier exploration, for sorting and filtering by different metrics.

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances
category,,,,,,,
Effective Date,291,0.20,4.0,58.79,90.99,1.55,255
Parties,1586,0.00,2.0,22.62,24.30,1.07,325
Non-Compete,164,0.60,43.5,392.77,377.03,0.96,77
Revenue/Profit Sharing,260,0.46,44.5,355.33,315.65,0.89,105
Insurance,341,0.39,39.0,322.96,286.58,0.89,107
Renewal Term,141,0.59,43.0,307.17,267.53,0.87,119
Anti-Assignment,404,0.18,34.5,302.33,256.15,0.85,235
Covenant Not To Sue,109,0.71,46.0,398.45,328.08,0.82,61
Warranty Duration,122,0.69,39.0,290.48,238.53,0.82,49


### Comparing near-duplicate categories

In [28]:
# restraint-of-trade categories
category_stats.loc[['Exclusivity', 'Non-Compete']].round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances
category,,,,,,,
Exclusivity,262,0.44,44.0,338.25,221.88,0.66,116
Non-Compete,164,0.60,43.5,392.77,377.03,0.96,77


In [29]:
# liability categories
category_stats.loc[['Cap On Liability', 'Uncapped Liability']].round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances
category,,,,,,,
Cap On Liability,432,0.25,48.0,363.50,230.5,0.63,181
Uncapped Liability,115,0.68,65.0,434.71,228.9,0.53,78


In [32]:
# license cateogires
category_stats.loc[['License Grant', 'Non-Transferable License']].round(2)

,n_answered,impossible_rate,words_median,len_mean,len_std,len_cv,positive_contract_appearances
category,,,,,,,
License Grant,513,0.24,52.0,423.93,294.12,0.69,164
Non-Transferable License,210,0.53,50.0,428.80,330.08,0.77,90


**TODO: finish updating this with train-only findings**
### Reading the tables

We can apply fairly similar conclusions from the original analysis on both training and validation data to the pure training split. This indicates a robust and balanced split.

- **Class balance**: We still prioritze categories with a high number of answerable questions (positive excerpts). We can safely rule out the 14 categories that contain either less than 100 examples or less than 20 positive contract appearances. 

- **Label consistency (`len_cv`)**: 
    - `Effective Date` (~1.5) and `Source Code Escrow` (~1.4) have highly inconsistent span lengths,suggesting inconsistent annotation granularity. Data from the training split has higher annotation variation than the combined train-validation set, which is expected given a smaller number of documents.
    - `Governing Law`, `Uncapped Liability`, and `Agreement` are the most consistent (lowest CV) categories with over 100 answerable examples.
- **Span difficulty (`words_median`)**: 
    - `Parties`, `Document Name`, `Agreement Date`, `Effective Date` are trivially short (2-4 words), which are simple to extract but not very representative of a typical legal clause. 
    - `License Grant`, `Cap On Liability`, `Audit Rights`, `Post-Termination Services` ...
- **Redundancy**: among the viable candidates, avoid picking more than one from the same cluster. Overlapping categories include:
    - Trade restraint: `Exclusivity` / `Non-Compete` / `Competitive Restriction Exception`
        - `Exclusivity` has the most answerable questions, lowest impossibility rate, and lowest coefficient of variance, with `Non-Compete` as the second strongest representative.
    - Liability: `Cap On Liability` / `Uncapped Liability`
        - `Cap On Liability` has a slightly higher coefficient of variance (0.67 vs 0.55), but it has a much higher number of answerable examples and much lower impossibility rate, making it a better candidate overall.
    - Licenses: `License Grant` / `Non-Transferable License` / `Affiliate License-Licensor`
        - `License Grant` has the highest number of answerables, the lowest impossibility, and the lowest coefficient of variance. `Non-Transferable License` has the 



### Proposed 10 core categories, grouped by bundles:
In no order of importance:

#### Bundle 1: Contract Metadata
Characterized by a high number of answerable examples and zero impossibility rate within the training split.\
**Categories**
- Parties
- Document Name

#### Bundle 2: Dates
**Categories**
- Agreement Date
- Expiration Date

#### Bundle 3: Licensing
**Categories**
- License Grant
- Non-Transferable License

#### Bundle 4: Trade Restraint
**Categories**
- Exclusivity
- Non-Compete

#### Bundle 5: Liability
**Categories**
- Cap on Liability
- Uncapped Liability